# EN4554 Assignment 2 — Convolutional Neural Networks

**Name:** Sadaruwan I J M J<br>
**Index number:** 220542J<br>
**GitHub repository:** _https://github.com/your-username/your-repo_

## 1. Setup

Imports, device selection and fixed random seeds so that results are reproducible.

In [1]:
import os, random, time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import CIFAR10

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True   # faster convolutions for fixed input sizes

print("PyTorch    :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("Device     :", device, f"({torch.cuda.get_device_name(0)})" if device.type == "cuda" else "")

PyTorch    : 2.11.0+cu128
torchvision: 0.26.0+cu128
Device     : cuda (NVIDIA GeForce RTX 5060 Laptop GPU)


## 1.1 Data: CIFAR-10

CIFAR-10 has 60,000 colour images of size 32×32 in 10 classes (50,000 training and 10,000 test images). It is downloaded to `./data` with `torchvision.datasets.CIFAR10` the first time this cell runs.

As the assignment requires:
- The 50k training images are split into **45k train / 5k validation** with a fixed seed.
- Images are **normalized per channel using statistics of the 45k training split only**.
- The **test set is used only once**, in Task 4.

In [ ]:
DATA_DIR = "./data"

train_full = CIFAR10(root=DATA_DIR, train=True,  download=True)
test_set   = CIFAR10(root=DATA_DIR, train=False, download=True)
CLASSES = train_full.classes

# Raw images as tensors: uint8 (N, 32, 32, 3) -> float in [0, 1], shape (N, 3, 32, 32)
def to_tensor(ds):
    x = torch.from_numpy(ds.data).permute(0, 3, 1, 2).float() / 255.0
    y = torch.tensor(ds.targets, dtype=torch.long)
    return x, y

X_full, y_full = to_tensor(train_full)
X_test, y_test = to_tensor(test_set)

# 45k / 5k train/validation split with a fixed seed
perm = torch.randperm(len(X_full), generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = perm[:45_000], perm[45_000:]
X_train, y_train = X_full[train_idx], y_full[train_idx]
X_val,   y_val   = X_full[val_idx],   y_full[val_idx]

# Per-channel normalization statistics from the training split only
MEAN = X_train.mean(dim=(0, 2, 3))
STD  = X_train.std(dim=(0, 2, 3))

def normalize(x):
    return (x - MEAN.view(1, 3, 1, 1).to(x.device)) / STD.view(1, 3, 1, 1).to(x.device)

print(f"Train: {tuple(X_train.shape)}  Val: {tuple(X_val.shape)}  Test: {tuple(X_test.shape)}")
print("Channel mean:", MEAN.numpy().round(4))
print("Channel std :", STD.numpy().round(4))
print("Train class counts:", torch.bincount(y_train).tolist())
print("Val class counts  :", torch.bincount(y_val).tolist())

In [ ]:
# A few training images with their labels
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, i in zip(axes.flat, range(16)):
    ax.imshow(X_train[i].permute(1, 2, 0).numpy())
    ax.set_title(CLASSES[y_train[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("CIFAR-10 training samples")
plt.tight_layout()
plt.show()

## Task 1: Backpropagation

## Task 2: Optimizers

## Task 3: Regularization and residual connections

## Task 4: CNN design (optional)